# EDA — Churn data trên MinIO

Notebook này kết nối MinIO, đọc các bảng silver, rồi xem schema, missing, phân bố thời gian và giá trị.

**Nguồn mặc định:** `lqminh/silver/devdb/`

Không dùng `churn_order_items`, `churn_products` (recommendation, không phải inactivity churn).
`churn_customer_snapshot` nằm bronze `test-doanh` và list rất chậm — panel `customer_id × snapshot_month` sẽ được dựng ở notebook featuring.

In [1]:
import sys
from pathlib import Path

import pandas as pd

for _p in [Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent]:
    if (_p / "minio_io.py").exists():
        sys.path.insert(0, str(_p.resolve()))
        break

from minio_io import DEFAULT_BUCKET, DEFAULT_PREFIX, TABLES, get_client, load_silver_tables

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## 1. Kết nối MinIO

In [2]:
client = get_client()
print("endpoint connected")
print("buckets:")
for bucket in client.list_buckets():
    print(" -", bucket.name)

endpoint connected
buckets:
 - doanhld
 - lqminh
 - minhmn
 - test-doanh


In [3]:
print(f"silver tables under {DEFAULT_BUCKET}/{DEFAULT_PREFIX}")
for obj in client.list_objects(DEFAULT_BUCKET, prefix=DEFAULT_PREFIX, recursive=False):
    print(" -", obj.object_name)

silver tables under lqminh/silver/devdb/
 - silver/devdb/churn_customers/
 - silver/devdb/churn_marketing_interactions/
 - silver/devdb/churn_orders/
 - silver/devdb/churn_payments/
 - silver/devdb/churn_product_usage/
 - silver/devdb/churn_subscriptions/
 - silver/devdb/churn_support_tickets/


## 2. Load toàn bộ bảng cần cho churn

Bỏ qua file Spark rác (`.spark-staging`, `_SUCCESS`, `_delta_log`).

In [4]:
tables = load_silver_tables(client)
print("\nloaded", list(tables))

loaded churn_customers: files=2, rows=10,002, cols=9
loaded churn_orders: files=37, rows=118,839, cols=6
loaded churn_payments: files=37, rows=199,916, cols=8
loaded churn_product_usage: files=38, rows=770,082, cols=6
loaded churn_subscriptions: files=1, rows=12,615, cols=8
loaded churn_support_tickets: files=36, rows=23,600, cols=7
loaded churn_marketing_interactions: files=37, rows=186,884, cols=7

loaded ['churn_customers', 'churn_orders', 'churn_payments', 'churn_product_usage', 'churn_subscriptions', 'churn_support_tickets', 'churn_marketing_interactions']


## 3. Hàm EDA

In [5]:
DATE_HINTS = (
    "date", "_at", "created", "sent_at", "start_date", "end_date",
    "birth_date", "signup_date", "closed_date", "last_login",
)


def infer_date_cols(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            cols.append(col)
        elif any(h in col.lower() for h in DATE_HINTS):
            cols.append(col)
    return cols


def eda_table(name: str, df: pd.DataFrame) -> None:
    print("=" * 72)
    print(name, "shape=", df.shape)
    print("-" * 72)
    print(df.dtypes.to_string())
    print("\nnull %")
    null_pct = (df.isna().mean() * 100).round(2)
    print(null_pct.to_string())
    print("\nhead")
    display(df.head(3))

    for col in infer_date_cols(df):
        s = pd.to_datetime(df[col], errors="coerce")
        print(f"date {col}: min={s.min()} max={s.max()} null={s.isna().sum()}")

    if "customer_id" in df.columns:
        print("nunique customer_id=", df["customer_id"].nunique())

    cat_cols = [c for c in df.columns if df[c].dtype == "object" or str(df[c].dtype) == "bool"]
    for col in cat_cols:
        vc = df[col].value_counts(dropna=False).head(10)
        print(f"\nvalue_counts {col}")
        print(vc.to_string())

## 4. EDA từng bảng

In [6]:
for name in TABLES:
    eda_table(name, tables[name])

churn_customers shape= (10002, 9)
------------------------------------------------------------------------
customer_id                int32
gender                    object
birth_date        datetime64[ns]
region                    object
city                      object
signup_date       datetime64[ns]
account_status            object
closed_date       datetime64[ns]
last_login_at     datetime64[ns]

null %
customer_id        0.00
gender             0.00
birth_date         0.00
region             0.00
city               0.00
signup_date        0.00
account_status     0.00
closed_date       93.08
last_login_at      0.00

head


,customer_id,gender,birth_date,region,city,signup_date,account_status,closed_date,last_login_at
0,1,M,1991-05-19,Mien Nam,Ho Chi Minh,2025-03-24,Active,NaT,2026-07-26 09:27:00
1,2,M,1979-12-31,Mien Bac,Quang Ninh,2025-05-15,Active,NaT,2025-11-21 03:19:00
2,3,M,1987-07-07,Mien Nam,Can Tho,2025-12-16,Active,NaT,2026-07-02 17:00:00


date birth_date: min=1961-08-15 00:00:00 max=2008-08-01 00:00:00 null=0
date signup_date: min=1970-01-01 00:00:00 max=2026-07-30 00:00:00 null=0
date closed_date: min=2023-09-02 00:00:00 max=2026-07-28 00:00:00 null=9310
date last_login_at: min=2023-08-27 05:02:00 max=2026-07-30 10:15:00 null=0
nunique customer_id= 10002

value_counts gender
gender
F    5032
M    4970

value_counts region
region
Mien Nam      3341
Mien Bac      3336
Mien Trung    3325

value_counts city
city
Nam Dinh       696
Bien Hoa       694
Da Nang        687
Can Tho        686
Hue            678
Vung Tau       675
Ha Noi         670
Ho Chi Minh    660
Quang Ninh     658
Quy Nhon       657

value_counts account_status
account_status
Active      9308
Closed       692
Inactive       2
churn_orders shape= (118839, 6)
------------------------------------------------------------------------
order_id                   int32
customer_id                int32
order_date        datetime64[ns]
status                    objec

,order_id,customer_id,order_date,status,total_amount,payment_method
0,172,23,2023-10-18 15:00:00,Returned,1496000.0,COD
1,214,26,2023-10-22 08:00:00,Completed,4910000.0,BankTransfer
2,215,26,2023-10-11 18:00:00,Completed,1190000.0,BankTransfer


date order_date: min=2023-08-01 10:00:00 max=2026-07-29 22:00:00 null=0
nunique customer_id= 9445

value_counts status
status
Completed    98580
Returned     11825
Cancelled     8434

value_counts payment_method
payment_method
COD             29791
BankTransfer    29741
CreditCard      29677
EWallet         29630
churn_payments shape= (199916, 8)
------------------------------------------------------------------------
payment_id                  int32
customer_id                 int32
order_id                  float64
subscription_id           float64
payment_date       datetime64[ns]
amount                    float32
status                     object
method                     object

null %
payment_id          0.00
customer_id         0.00
order_id           33.19
subscription_id    66.81
payment_date        0.00
amount              0.00
status              0.00
method              0.00

head


...

date start_date: min=2023-07-30 00:00:00 max=2026-07-28 00:00:00 null=0
date end_date: min=2023-09-02 00:00:00 max=2026-07-28 00:00:00 null=7235
nunique customer_id= 10000

value_counts plan_tier
plan_tier
Free       6117
Plus       4383
Premium    2115

value_counts change_type
change_type
New          10000
Upgrade       1720
Downgrade      895

value_counts auto_renew
auto_renew
True     9850
False    2765

value_counts status
status
Active       9850
Cancelled    1690
Expired      1075
churn_support_tickets shape= (23600, 7)
------------------------------------------------------------------------
ticket_id                    int32
customer_id                  int32
created_at          datetime64[ns]
category                    object
priority                    object
resolution_hours           float32
csat_score                 float64

null %
ticket_id            0.00
customer_id          0.00
created_at           0.00
category             0.00
priority             0.00
resolutio

,ticket_id,customer_id,created_at,category,priority,resolution_hours,csat_score
0,188,73,2023-10-06 04:00:00,Other,Medium,4.3,3.0
1,273,111,2023-10-25 02:00:00,Shipping,Low,5.5,NaN
2,543,220,2023-10-19 01:00:00,Account,Medium,2.6,4.0


date created_at: min=2023-08-07 11:00:00 max=2026-07-28 23:00:00 null=0
nunique customer_id= 7667

value_counts category
category
Shipping    4817
Product     4733
Other       4704
Account     4682
Payment     4664

value_counts priority
priority
Low       11772
Medium     8221
High       3607
churn_marketing_interactions shape= (186884, 7)
------------------------------------------------------------------------
interaction_id             int32
customer_id                int32
sent_at           datetime64[ns]
opened                      bool
clicked                     bool
converted                   bool
channel                   object

null %
interaction_id    0.0
customer_id       0.0
sent_at           0.0
opened            0.0
clicked           0.0
converted         0.0
channel           0.0

head


,interaction_id,customer_id,sent_at,opened,clicked,converted,channel
0,445,23,2023-10-04 16:00:00,False,False,False,Email
1,450,23,2023-10-18 14:00:00,False,False,False,Email
2,496,26,2023-10-13 14:00:00,False,False,False,SMS


date sent_at: min=2023-07-30 10:00:00 max=2026-07-28 20:00:00 null=0
nunique customer_id= 9973

value_counts opened
opened
False    110695
True      76189

value_counts clicked
clicked
False    160014
True      26870

value_counts converted
converted
False    184741
True       2143

value_counts channel
channel
Email    111945
SMS       46628
Push      28311


## 4b. Đếm missing value

`isna()` bắt `NaN` / `NaT` / `None`. Cột string/`object` còn có thể là `""` — pandas **không** coi chuỗi rỗng là missing, nên đếm riêng.

In [7]:
missing_rows = []
for name, df in tables.items():
    n = len(df)
    for col in df.columns:
        n_null = int(df[col].isna().sum())
        if df[col].dtype == "object" or str(df[col].dtype) == "string":
            n_blank = int(df[col].fillna("").astype(str).str.strip().eq("").sum() - n_null)
            n_blank = max(n_blank, 0)
        else:
            n_blank = 0
        n_missing = n_null + n_blank
        missing_rows.append(
            {
                "table": name,
                "column": col,
                "n_null": n_null,
                "n_blank_string": n_blank,
                "n_missing": n_missing,
                "pct_missing": round(100 * n_missing / n, 2) if n else 0.0,
                "n_rows": n,
            }
        )

missing_df = pd.DataFrame(missing_rows)
print("Cột có missing (null hoặc chuỗi rỗng)")
display(missing_df.loc[missing_df["n_missing"] > 0].sort_values(["table", "pct_missing"], ascending=[True, False]))

print("\nTổng missing theo bảng")
display(
    missing_df.groupby("table", as_index=False)
    .agg(n_rows=("n_rows", "first"), n_null=("n_null", "sum"), n_blank_string=("n_blank_string", "sum"))
)

print("\nToàn bộ cột (kể cả 0 missing)")
display(missing_df)

Cột có missing (null hoặc chuỗi rỗng)


,table,column,n_null,n_blank_string,n_missing,pct_missing,n_rows
7,churn_customers,closed_date,9310,0,9310,93.08,10002
18,churn_payments,subscription_id,133570,0,133570,66.81,199916
17,churn_payments,order_id,66346,0,66346,33.19,199916
34,churn_subscriptions,end_date,7235,0,7235,57.35,12615
43,churn_support_tickets,csat_score,7022,0,7022,29.75,23600



Tổng missing theo bảng


,table,n_rows,n_null,n_blank_string
0,churn_customers,10002,9310,0
1,churn_marketing_interactions,186884,0,0
2,churn_orders,118839,0,0
3,churn_payments,199916,199916,0
4,churn_product_usage,770082,0,0
5,churn_subscriptions,12615,7235,0
6,churn_support_tickets,23600,7022,0



Toàn bộ cột (kể cả 0 missing)


,table,column,n_null,n_blank_string,n_missing,pct_missing,n_rows
0,churn_customers,customer_id,0,0,0,0.00,10002
1,churn_customers,gender,0,0,0,0.00,10002
2,churn_customers,birth_date,0,0,0,0.00,10002
3,churn_customers,region,0,0,0,0.00,10002
4,churn_customers,city,0,0,0,0.00,10002
5,churn_customers,signup_date,0,0,0,0.00,10002
6,churn_customers,account_status,0,0,0,0.00,10002
7,churn_customers,closed_date,9310,0,9310,93.08,10002
8,churn_customers,last_login_at,0,0,0,0.00,10002
9,churn_orders,order_id,0,0,0,0.00,118839


## Vì sao string hiện thành `object`?

Đó **không phải lỗi convert**. Pandas mặc định map chuỗi Python thành dtype `object` (cột chứa pointer tới `str` trong RAM).

- Parquet/Spark có kiểu `string` / UTF8.
- `pd.read_parquet()` (pyarrow → pandas) thường **downcast** string thành `object`, trừ khi bật pandas StringDtype (`string`).
- `object` = hộp chứa bất kỳ Python object nào: `str`, `None`, đôi khi lẫn số. `df.dtypes` vì vậy in `object`, không in `str`.

Muốn thấy `string` (nullable, Arrow-backed):

```python
df["gender"] = df["gender"].astype("string")
# hoặc khi đọc:
pd.read_parquet(buf, dtype_backend="pyarrow")
```

Với EDA/feature hiện tại, để `object` là bình thường; `gender`, `region`, `status` vẫn là text.

## 5. Phủ khách hàng giữa các bảng

Spine train sẽ là `churn_customers`. Các bảng event left-join theo `customer_id`.

In [8]:
customers = tables["churn_customers"]
base_ids = set(customers["customer_id"])
print("customers", len(base_ids))
print(customers["account_status"].value_counts(dropna=False).to_string())
signup = pd.to_datetime(customers["signup_date"], errors="coerce")
print("signup year")
print(signup.dt.year.value_counts(dropna=False).sort_index().to_string())
print("signup < 2020 (sentinel, drop khi featuring)", int((signup < "2020-01-01").sum()))
print()
for name, df in tables.items():
    if name == "churn_customers" or "customer_id" not in df.columns:
        continue
    ids = set(df["customer_id"])
    print(
        f"{name:34} rows={len(df):7,}  customers={len(ids):5,}  "
        f"in_customers={len(ids & base_ids):5,}  not_in_customers={len(ids - base_ids):5,}"
    )

customers 10002
account_status
Active      9308
Closed       692
Inactive       2
signup year
signup_date
1970       1
2023    1503
2024    3413
2025    3381
2026    1704
signup < 2020 (sentinel, drop khi featuring) 1

churn_orders                       rows=118,839  customers=9,445  in_customers=9,445  not_in_customers=    0
churn_payments                     rows=199,916  customers=9,545  in_customers=9,545  not_in_customers=    0
churn_product_usage                rows=770,082  customers=10,000  in_customers=10,000  not_in_customers=    0
churn_subscriptions                rows= 12,615  customers=10,000  in_customers=10,000  not_in_customers=    0
churn_support_tickets              rows= 23,600  customers=7,667  in_customers=7,667  not_in_customers=    0
churn_marketing_interactions       rows=186,884  customers=9,973  in_customers=9,973  not_in_customers=    0


## 6. Ghi chú cho bước featuring

- `last_login_at` trên `churn_customers` là giá trị **hiện tại** → không dùng trực tiếp cho mọi snapshot (leak). Recency tính từ event `<= t`.
- `product_usage.event_type` đã có `Login` → activity = usage + order là đủ cho nhãn inactivity.
- Nhãn `churn_30d` / `churn_60d`: không có activity trong `(t, t+30/60]`.
- Bước tiếp: chạy `02_build_feature_dataset.ipynb`.